# 01 — Data Cleaning

**Goal:** produce four clean tables — `vendors_clean`, `products_clean`,
`purchase_orders_clean`, `quality_inspections_clean` — from the raw CSVs, and
log every cleaning decision in plain English so the interviewer can defend
every number.

**Cleaning decisions made in this notebook:**
1. Vendor-name casing — two vendor names had inconsistent casing; normalised
   with `.str.title()`.
2. Duplicate POs — the raw file contains a handful of exact duplicates
   (accidental double-entry). Drop on `po_number`, keep first.
3. Missing `actual_delivery_date` (~1%) — a legitimate "in transit" state.
   Kept as `NaT`; excluded from the OTD denominator downstream.
4. `agreed_price = 0` or `NULL` (< 1%) — flagged and excluded from price
   variance.
5. `quantity_received > quantity_ordered` — realistic over-shipments. Left
   truthfully in the data; fill rate is capped at 100 % in the SQL layer and
   these rows are flagged in a DQ view.
6. Missing `defect_count` (~2 %) — filled with 0 only when
   `quality_status = 'Passed'`. Otherwise left `NaN` (so the row is excluded
   from defect calculations rather than silently zeroed).
7. Inspection coverage — only ~60 % of POs get inspected. This is a business
   fact, not a data-quality problem. Documented explicitly so the interviewer
   sees the LEFT-JOIN choice below is intentional.


## Profiling — shape, dtypes, nulls, coverage


In [1]:
import pandas as pd
from pathlib import Path

RAW = Path.cwd().parent / "data" / "raw"

v  = pd.read_csv(RAW / "vendors.csv",             parse_dates=["contract_start_date"])
p  = pd.read_csv(RAW / "products.csv")
po = pd.read_csv(RAW / "purchase_orders.csv",
                 parse_dates=["order_date", "promised_date", "actual_delivery_date"])
qi = pd.read_csv(RAW / "quality_inspections.csv", parse_dates=["inspection_date"])

for name, df in {"vendors": v, "products": p,
                 "purchase_orders": po, "quality_inspections": qi}.items():
    print(f"--- {name} ---")
    print(f"  shape:      {df.shape}")
    print(f"  duplicates: {df.duplicated().sum()}")
    print(f"  nulls:      {df.isna().sum().to_dict()}")
    print()

# Coverage
insp_pos = qi['po_number'].nunique()
uniq_pos = po['po_number'].nunique()
print(f"Inspection coverage: {insp_pos}/{uniq_pos} = {100*insp_pos/uniq_pos:.1f}%")


--- vendors ---
  shape:      (20, 6)
  duplicates: 0
  nulls:      {'vendor_id': 0, 'vendor_name': 0, 'region': 0, 'vendor_category': 0, 'contract_start_date': 0, 'payment_terms': 0}

--- products ---
  shape:      (16, 4)
  duplicates: 0
  nulls:      {'product_id': 0, 'product_name': 0, 'category': 0, 'standard_unit_price': 0}

--- purchase_orders ---
  shape:      (1410, 12)
  duplicates: 10
  nulls:      {'po_number': 0, 'vendor_id': 0, 'product_id': 0, 'order_date': 0, 'promised_date': 0, 'actual_delivery_date': 11, 'quantity_ordered': 0, 'quantity_received': 0, 'unit_price': 0, 'agreed_price': 0, 'invoice_amount': 0, 'order_status': 0}

--- quality_inspections ---
  shape:      (840, 7)
  duplicates: 0
  nulls:      {'inspection_id': 0, 'po_number': 0, 'inspection_date': 0, 'inspected_quantity': 0, 'defect_count': 10, 'complaint_count': 0, 'quality_status': 0}

Inspection coverage: 840/1400 = 60.0%


## Apply the cleaning pipeline

The full pipeline lives in `scripts/cleaning.py`. We call it here so the
notebook prints the decision log inline.


In [2]:
import sys
sys.path.insert(0, str(Path.cwd().parent / "scripts"))
from cleaning import load_raw, clean, save_to_csv, save_to_db

raw     = load_raw()
cleaned, log = clean(raw)
save_to_csv(cleaned)
save_to_db(cleaned)   # writes *_clean tables to Postgres

print("Cleaning decision log:")
for k, val in log.items():
    print(f"  {k:35s}  {val}")


Cleaning decision log:
  vendor_name_casing_fixed             3
  duplicate_pos_dropped                10
  agreed_price_zero_or_null            5
  missing_actual_delivery              11
  overshipments_flagged                372
  defect_count_filled_zero             3
  defect_count_still_missing           7


## Post-clean sanity checks


In [3]:
po_c = cleaned['purchase_orders']
qi_c = cleaned['quality_inspections']
v_c  = cleaned['vendors']

print("Vendors after casing fix:")
print(v_c['vendor_name'].tolist())

print(f"\nPOs after de-dup: {len(po_c)}")
print(f"Missing actual_delivery_date: {po_c['actual_delivery_date'].isna().sum()}")
print(f"Over-shipments still flagged: {(po_c['quantity_received'] > po_c['quantity_ordered']).sum()}")
print(f"Inspected POs: {len(qi_c)}")
print(f"Missing defect_count (should be Failed/Conditional only): {qi_c['defect_count'].isna().sum()}")


Vendors after casing fix:
['Bharat Packaging Co', 'Ganga Industries', 'Krishna Traders', 'Meghna Supplies', 'Sundar Logistics', 'Ravi Raw Materials', 'Vikram Enterprises', 'Aditya Corp', 'Sagar Packworks', 'Himalaya Mro', 'Chennai Freight Ltd', 'Kolkata Rawmet', 'Deccan Distributors', 'Ashoka Papers', 'Nilgiri Metals', 'Konark Cargo', 'Godavari Supplies', 'Yamuna Packaging', 'Narmada Traders', 'Kaveri Movers']

POs after de-dup: 1400
Missing actual_delivery_date: 11
Over-shipments still flagged: 372
Inspected POs: 840
Missing defect_count (should be Failed/Conditional only): 7
